In [1]:
# ═══════════════════════════════════════════════════════════════════════════════
# NOTEBOOK 06: DATA CLEANING & FEATURE ENGINEERING
# Purpose: Handle missing values, create new features, and prepare clean data
# ═══════════════════════════════════════════════════════════════════════════════

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("Set2")
pd.set_option('display.max_columns', None)

# ═══════════════════════════════════════════════════════════════════════════════
# 1. LOAD DATA
# ═══════════════════════════════════════════════════════════════════════════════

df = pd.read_csv('../data/raw/customer_churn_dataset.csv')

print("="*80)
print("DATA CLEANING & FEATURE ENGINEERING")
print("="*80)
print(f"\nOriginal Dataset: {df.shape[0]:,} rows × {df.shape[1]} columns\n")

# ═══════════════════════════════════════════════════════════════════════════════
# 2. MISSING VALUES ANALYSIS & HANDLING
# ═══════════════════════════════════════════════════════════════════════════════

print("="*80)
print("MISSING VALUES ANALYSIS")
print("="*80)

missing_stats = pd.DataFrame({
    'Column': df.columns,
    'Missing_Count': df.isnull().sum().values,
    'Missing_Percent': (df.isnull().sum().values / len(df) * 100).round(2)
})

missing_stats = missing_stats[missing_stats['Missing_Count'] > 0].sort_values('Missing_Count', ascending=False)

if len(missing_stats) > 0:
    print("\nColumns with missing values:")
    print(missing_stats.to_string(index=False))
else:
    print("\n✓ No missing values found!")

# ═══════════════════════════════════════════════════════════════════════════════
# 3. IMPUTATION STRATEGY (WITHOUT DATA LEAKAGE)
# ═══════════════════════════════════════════════════════════════════════════════

print("\n" + "="*80)
print("APPLYING IMPUTATION STRATEGY")
print("="*80)

if 'TotalCharges' in df.columns and df['TotalCharges'].isnull().sum() > 0:
    df['TotalCharges'] = df.apply(
        lambda row: row['MonthlyCharges'] * row['TenureMonths'] 
        if pd.isnull(row['TotalCharges']) and row['TenureMonths'] <= 2
        else row['TotalCharges'], 
        axis=1
    )
    if df['TotalCharges'].isnull().sum() > 0:
        df['TotalCharges'] = df.groupby(['ContractType', pd.cut(df['TenureMonths'], bins=[-1, 12, 24, 48, 100])])['TotalCharges'].transform(
            lambda x: x.fillna(x.median())
        )
    if df['TotalCharges'].isnull().sum() > 0:
        df['TotalCharges'].fillna(df['TotalCharges'].median(), inplace=True)
    print("  ✓ TotalCharges imputed")

# FIX: Removed 'Churn' from groupby to prevent data leakage
if 'SatisfactionScore' in df.columns and df['SatisfactionScore'].isnull().sum() > 0:
    df['SatisfactionScore'] = df.groupby(['ContractType'])['SatisfactionScore'].transform(
        lambda x: x.fillna(x.median())
    )
    if df['SatisfactionScore'].isnull().sum() > 0:
        df['SatisfactionScore'].fillna(df['SatisfactionScore'].median(), inplace=True)
    print("  ✓ SatisfactionScore imputed")

for col in df.columns:
    if df[col].isnull().sum() > 0:
        if df[col].dtype in ['int64', 'float64']:
            df[col].fillna(df[col].median(), inplace=True)
        else:
            df[col].fillna(df[col].mode()[0], inplace=True)
        print(f"  ✓ Catch-all imputation applied to {col}")

# ═══════════════════════════════════════════════════════════════════════════════
# 4. BUSINESS-DRIVEN FEATURES (WITHOUT SCALING - SCALING MOVED TO MODELING)
# ═══════════════════════════════════════════════════════════════════════════════

print("\n" + "="*80)
print("CREATING BUSINESS-DRIVEN FEATURES")
print("="*80)

# Basic financial metrics
df['ARPU'] = df['TotalCharges'] / (df['TenureMonths'] + 1)
df['ChargesRatio'] = df['MonthlyCharges'] / (df['TotalCharges'] + 1)

# FIX: Removed redundant CLV_Proxy (exact copy of TotalCharges)
# Engagement, Risk, and Satisfaction scores will be created WITHOUT normalization
# Normalization will happen in modeling notebook after train/test split

df['EngagementScore_Raw'] = (
    df['AvgMonthlyUsageGB'] * 0.4 +
    df['NumProducts'] * 0.3 +
    (100 - df['DaysSinceLastInteraction']) * 0.3
)

df['RiskScore_Raw'] = (
    df['SupportCalls'] * 0.3 +
    df['LatePayments'] * 0.4 +
    df['DaysSinceLastInteraction'] * 0.3
)

df['SatisfactionIndex_Raw'] = (
    df['SatisfactionScore'] * 0.6 +
    (10 - df['SupportCalls']) * 0.2 +
    (10 - df['LatePayments']) * 0.2
)

# Categorical features
df['TenureCategory'] = pd.cut(df['TenureMonths'], bins=[0, 12, 24, 48, 1000], labels=['0-1 Year', '1-2 Years', '2-4 Years', '4+ Years'], include_lowest=True)
df['AgeGroup'] = pd.cut(df['Age'], bins=[0, 25, 35, 45, 55, 1000], labels=['18-25', '26-35', '36-45', '46-55', '56+'], include_lowest=True)
df['ChargesCategory'] = pd.cut(df['MonthlyCharges'], bins=[0, 30, 60, 90, 1000], labels=['Low', 'Medium', 'High', 'Premium'], include_lowest=True)
df['UsageIntensity'] = pd.cut(df['AvgMonthlyUsageGB'], bins=[0, 10, 30, 60, 1000], labels=['Light', 'Moderate', 'Heavy', 'Power User'], include_lowest=True)

# Service adoption
service_cols = ['PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 
                'OnlineBackup', 'TechSupport', 'StreamingTV', 'StreamingMovies']
df['ServiceAdoptionScore'] = df[service_cols].apply(lambda x: sum([1 for val in x if val not in ['No', 'No internet service']]), axis=1)

# Payment and interaction metrics
df['PaymentReliability'] = 100 - (df['LatePayments'] * 10)
df['InteractionRecency'] = 100 - df['DaysSinceLastInteraction']

# Contract value
contract_mapping = {'Month-to-month': 1, 'One year': 2, 'Two year': 3}
df['ContractValueScore'] = df['ContractType'].map(contract_mapping).fillna(1)

# Premium and streaming services
df['HasPremiumServices'] = ((df['OnlineSecurity'] == 'Yes') | (df['OnlineBackup'] == 'Yes') | (df['TechSupport'] == 'Yes')).astype(int)
df['HasStreamingServices'] = ((df['StreamingTV'] == 'Yes') | (df['StreamingMovies'] == 'Yes')).astype(int)

# Additional metrics
df['TotalServicesCount'] = df['NumProducts'] + df['ServiceAdoptionScore']
df['ChargesPerProduct'] = df['MonthlyCharges'] / (df['NumProducts'] + 1)
df['SupportIntensity'] = df['SupportCalls'] / (df['TenureMonths'] + 1)
df['LatePaymentRate'] = df['LatePayments'] / (df['TenureMonths'] + 1)

# Fill any NaN in new features
new_features = ['ARPU', 'ChargesRatio', 'EngagementScore_Raw', 'RiskScore_Raw',
                'SatisfactionIndex_Raw', 'ServiceAdoptionScore', 'PaymentReliability',
                'InteractionRecency', 'ContractValueScore', 'HasPremiumServices',
                'HasStreamingServices', 'TotalServicesCount', 'ChargesPerProduct',
                'SupportIntensity', 'LatePaymentRate']
df[new_features] = df[new_features].fillna(0)

for cat_col in ['TenureCategory', 'AgeGroup', 'ChargesCategory', 'UsageIntensity']:
    if df[cat_col].isnull().sum() > 0:
        df[cat_col] = df[cat_col].fillna(df[cat_col].mode()[0])

print(f"\n✓ Total new features created: 15")
print(f"✓ Dataset shape after feature engineering: {df.shape[0]:,} rows × {df.shape[1]} columns")

# ═══════════════════════════════════════════════════════════════════════════════
# 5. ENCODING CATEGORICAL VARIABLES
# ═══════════════════════════════════════════════════════════════════════════════

print("\n" + "="*80)
print("ENCODING CATEGORICAL VARIABLES")
print("="*80)

df_encoded = df.copy()

# Encode target variable
df_encoded['Churn'] = df_encoded['Churn'].map({'Yes': 1, 'No': 0})
print("  ✓ Churn encoded to 0/1")

# Binary columns
binary_cols = ['PhoneService', 'PaperlessBilling']
for col in binary_cols:
    if col in df_encoded.columns:
        df_encoded[col] = df_encoded[col].map({'Yes': 1, 'No': 0}).fillna(0)
        print(f"  ✓ {col} encoded")

# Multi-level service columns
multi_level_cols = ['MultipleLines', 'OnlineSecurity', 'OnlineBackup', 'TechSupport', 'StreamingTV', 'StreamingMovies']
for col in multi_level_cols:
    if col in df_encoded.columns:
        df_encoded[col] = df_encoded[col].map({'Yes': 1, 'No': 0, 'No phone service': 0, 'No internet service': 0}).fillna(0)
        print(f"  ✓ {col} encoded")

# FIX: Use One-Hot Encoding for nominal variables (no inherent order)
nominal_cols = ['Gender', 'InternetService', 'PaymentMethod']
for col in nominal_cols:
    if col in df_encoded.columns:
        dummies = pd.get_dummies(df_encoded[col], prefix=col, drop_first=True)
        df_encoded = pd.concat([df_encoded, dummies], axis=1)
        df_encoded.drop(col, axis=1, inplace=True)
        print(f"  ✓ {col} one-hot encoded and original dropped")

# Ordinal encoding for ContractType (has natural order)
if 'ContractType' in df_encoded.columns:
    contract_order = {'Month-to-month': 0, 'One year': 1, 'Two year': 2}
    df_encoded['ContractType_Encoded'] = df_encoded['ContractType'].map(contract_order).fillna(0)
    df_encoded.drop('ContractType', axis=1, inplace=True)
    print("  ✓ ContractType ordinal encoded and original dropped")

# City one-hot encoding
if 'City' in df_encoded.columns:
    city_dummies = pd.get_dummies(df_encoded['City'], prefix='City', drop_first=True)
    df_encoded = pd.concat([df_encoded, city_dummies], axis=1)
    df_encoded.drop('City', axis=1, inplace=True)
    print("  ✓ City one-hot encoded and original dropped")

# Encode created categorical features
cat_feature_mapping = {
    'TenureCategory': {'0-1 Year': 0, '1-2 Years': 1, '2-4 Years': 2, '4+ Years': 3},
    'AgeGroup': {'18-25': 0, '26-35': 1, '36-45': 2, '46-55': 3, '56+': 4},
    'ChargesCategory': {'Low': 0, 'Medium': 1, 'High': 2, 'Premium': 3},
    'UsageIntensity': {'Light': 0, 'Moderate': 1, 'Heavy': 2, 'Power User': 3}
}

for col, mapping in cat_feature_mapping.items():
    if col in df_encoded.columns:
        df_encoded[col + '_Encoded'] = df_encoded[col].astype(str).map(mapping).fillna(0)
        df_encoded.drop(col, axis=1, inplace=True)
        print(f"  ✓ {col} ordinal encoded and original dropped")

# Drop CustomerID if exists
if 'CustomerID' in df_encoded.columns:
    df_encoded.drop('CustomerID', axis=1, inplace=True)
    print("  ✓ CustomerID dropped")

# ═══════════════════════════════════════════════════════════════════════════════
# 6. FINAL VALIDATION
# ═══════════════════════════════════════════════════════════════════════════════

print("\n" + "="*80)
print("FINAL VALIDATION")
print("="*80)

# Check for NaN
nan_count = df_encoded.isnull().sum().sum()
print(f"Total NaN values: {nan_count}")
if nan_count > 0:
    print("  ⚠ WARNING: Dataset still has NaN!")
    print(df_encoded.isnull().sum()[df_encoded.isnull().sum() > 0])
else:
    print("  ✓ Dataset is perfectly clean!")

# Check for non-numeric columns
non_numeric = df_encoded.select_dtypes(include=['object']).columns.tolist()
if len(non_numeric) > 0:
    print(f"\n⚠ WARNING: Non-numeric columns found: {non_numeric}")
else:
    print("\n✓ All columns are numeric!")

print(f"\nFinal shape: {df_encoded.shape[0]:,} rows × {df_encoded.shape[1]} columns")

# ═══════════════════════════════════════════════════════════════════════════════
# 7. SAVE PROCESSED DATA
# ═══════════════════════════════════════════════════════════════════════════════

df.to_csv('../data/processed/customer_churn_features.csv', index=False)
df_encoded.to_csv('../data/processed/customer_churn_encoded.csv', index=False)
print("\n✓ Processed datasets saved successfully!")
print("  - customer_churn_features.csv (with original categorical columns)")
print("  - customer_churn_encoded.csv (fully numeric, ready for modeling)")


DATA CLEANING & FEATURE ENGINEERING

Original Dataset: 70,000 rows × 26 columns

MISSING VALUES ANALYSIS

Columns with missing values:
           Column  Missing_Count  Missing_Percent
     TotalCharges           6953             9.93
SatisfactionScore           2121             3.03

APPLYING IMPUTATION STRATEGY
  ✓ TotalCharges imputed
  ✓ SatisfactionScore imputed

CREATING BUSINESS-DRIVEN FEATURES

✓ Total new features created: 15
✓ Dataset shape after feature engineering: 70,000 rows × 45 columns

ENCODING CATEGORICAL VARIABLES
  ✓ Churn encoded to 0/1
  ✓ PhoneService encoded
  ✓ PaperlessBilling encoded
  ✓ MultipleLines encoded
  ✓ OnlineSecurity encoded
  ✓ OnlineBackup encoded
  ✓ TechSupport encoded
  ✓ StreamingTV encoded
  ✓ StreamingMovies encoded
  ✓ Gender one-hot encoded and original dropped
  ✓ InternetService one-hot encoded and original dropped
  ✓ PaymentMethod one-hot encoded and original dropped
  ✓ ContractType ordinal encoded and original dropped
  ✓ City one-h